In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!pip install torchmetrics

In [ ]:
#!unzip /content/drive/MyDrive/SESM/PySESM.zip
!mkdir results_1
!mkdir results_2
!mkdir results_3
!mkdir results_1/plots
!mkdir results_2/plots
!mkdir results_3/plots
!mkdir results_1/stats
!mkdir results_2/stats
!mkdir results_3/stats

mkdir: cannot create directory ‘results_1’: File exists
mkdir: cannot create directory ‘results_2’: File exists
mkdir: cannot create directory ‘results_3’: File exists
mkdir: cannot create directory ‘results_1/plots’: File exists
mkdir: cannot create directory ‘results_2/plots’: File exists
mkdir: cannot create directory ‘results_3/plots’: File exists
mkdir: cannot create directory ‘results_1/stats’: File exists
mkdir: cannot create directory ‘results_2/stats’: File exists
mkdir: cannot create directory ‘results_3/stats’: File exists


In [ ]:
#!pip install wandb -qU

In [ ]:
import torch
import sys

# Agregar la ruta al sys.path
sys.path.append('/content/drive/MyDrive/SESM')
#sys.path.append('/content/drive/u/0/shared-with-me/SESM')

#import wandb
#from torchmetrics import MeanSquaredError
from sklearn.metrics import mean_squared_error

import numpy as np
from scipy.stats import multivariate_normal

from PySESM.models.SESM.SESM import SESM_Model
from PySESM.base_functions.Function import GaussianFunctions
from PySESM.models.ISTALayer import ISTALayer
from PySESM.utils.loggers import *

from sklearn.model_selection import train_test_split
from collections import defaultdict
import pandas as pd
import random
from google.colab import files
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go


In [ ]:
SEED=24

## Login de Wandb

In [ ]:
#wandb.login()

In [ ]:
logger = setup_logger()
logger.setLevel(logging.INFO)

## Definicion de covarianzas no diagnonales

In [ ]:

# Non-diagonal covariance
def generate_sigma_tensors():
    """
    Generates non-diagonal covariance tensors for 2D Gaussian distributions.

    Returns:
    tuple: Tuple containing three non-diagonal covariance tensors (sigma1, sigma2, sigma3).
    """
    e0 = torch.tensor([1.0, 1.0], dtype=torch.float32)
    e0 = e0 / e0.norm()

    def generate_sigma(rotation_angle, scaling_factors):
        rotation_matrix = torch.tensor([[np.cos(rotation_angle), -np.sin(rotation_angle)],
                                       [np.sin(rotation_angle), np.cos(rotation_angle)]], dtype=torch.float32)
        e = torch.mm(rotation_matrix, e0.unsqueeze(1))
        E = torch.cat((e0.unsqueeze(1), e), dim=1)
        D = torch.diag(torch.tensor(scaling_factors, dtype=torch.float32))
        return torch.mm(torch.mm(E, D), E.t())

    sigma1 = generate_sigma(np.pi/4, [0.4, 0.1])
    sigma2 = generate_sigma(np.pi/4, [0.05, 0.5])
    sigma3 = generate_sigma(np.pi/4, [0.2, 0.5])

    return sigma1, sigma2, sigma3


In [ ]:
sigma1, sigma2, sigma3 = generate_sigma_tensors()

## Definicion de varianzas diagonales

In [ ]:
sigma1_d = 0.15 * torch.eye(2)
sigma2_d = 0.2 * torch.eye(2)
sigma3_d = 0.3 *torch.eye(2)

## Crear Gaussianas

In [ ]:

def generate_z(X, sigma, mu):
    """
    Generates a combined density of three multivariate normal distributions evaluated at points X.

    Args:
    - X (torch.Tensor): A tensor containing points where the distributions are evaluated.
    - sigma (list of torch.Tensor): A list of covariance matrices for the distributions.
    - mu (list of torch.Tensor): A list of mean vectors for the distributions.

    Returns:
    - torch.Tensor: The combined density values of the three distributions at points X.
    """
    z1 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu[0].numpy(), sigma[0].numpy()), dtype=torch.float32)
    z2 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu[1].numpy(), sigma[1].numpy()), dtype=torch.float32)
    z3 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu[2].numpy(), sigma[2].numpy()), dtype=torch.float32)

    return z1 + z2 + z3

In [ ]:
def generate_mesh(n_points, xl, xr, sigma, mu):
  """
  Generates a 2D mesh grid and evaluates a combined density of three multivariate normal distributions on it.

  Args:
  - n_points (int): Number of points in each dimension of the grid.
  - xl (float): The lower bound of the grid.
  - xr (float): The upper bound of the grid.
  - sigma (list of torch.Tensor): A list of covariance matrices for the distributions.
  - mu (list of torch.Tensor): A list of mean vectors for the distributions.

  Returns:
  - xx (np.ndarray): The x-coordinates of the mesh grid.
  - yy (np.ndarray): The y-coordinates of the mesh grid.
  - zz (torch.Tensor): The combined density values of the three distributions on the mesh grid.
  """
  x = np.linspace(xl, xr, n_points)
  xx, yy = np.meshgrid(x, x)
  X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)
  zz = generate_z(X, sigma, mu)
  zz = zz.reshape(xx.shape)
  return xx, yy, zz

In [ ]:
def generate_random_samples(n_points, xl, xr, sigma, mu):
  """
  Generates random samples within a specified range and evaluates a combined density of three multivariate normal distributions on them.

  Args:
  - n_points (int): Number of random points to generate.
  - xl (float): The lower bound of the range for generating random points.
  - xr (float): The upper bound of the range for generating random points.
  - sigma (list of torch.Tensor): A list of covariance matrices for the distributions.
  - mu (list of torch.Tensor): A list of mean vectors for the distributions.
  - SEED (int, optional): Seed for random number generation to ensure reproducibility. Default is 42.

  Returns:
  - xx (np.ndarray): The x-coordinates of the random samples.
  - yy (np.ndarray): The y-coordinates of the random samples.
  - zz (torch.Tensor): The combined density values of the three distributions at the random sample points.
  """
  np.random.seed(SEED)
  xx = np.random.uniform(xl, xr, n_points)
  yy = np.random.uniform(xl, xr, n_points)

  X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)
  zz = generate_z(X, sigma, mu)

  return xx, yy, zz

In [ ]:
def generate_mu(x_center, y_center):
  """
  Generates a mean vector for a multivariate normal distribution.

  Args:
  - x_center (float): The x-coordinate of the center.
  - y_center (float): The y-coordinate of the center.

  Returns:
  - torch.Tensor: A tensor containing the mean vector [x_center, y_center].
  """
  return torch.tensor([x_center, y_center], dtype=torch.float32)

## Covarianza de gaussianas del experimento
  - 3 Diagonales
  - 2 Diagonales, 1 no diagonal
  - 1 diagonal, 2 no diagonales
  - 3 no diagonales

In [ ]:
mu1 = generate_mu(1, 1)
mu2 = generate_mu(1, -1)
mu3 = generate_mu(-1, -1)

#sigmas Non-diagonal covariance
sigma_list = [sigma1_d, sigma2_d, sigma3_d]

mu_list = [mu1,mu2,mu3]

xx, yy, zz = generate_mesh(50, -2, 2, sigma_list, mu_list)


xx_r, yy_r, zz_r = generate_random_samples(500, -2, 2, sigma_list, mu_list)

print("X: ", xx[:2])
print("Y: ", yy[:2])
print("Z: ", zz[:2])
# # 3 diag
# xx, yy, zz = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3_d)
# xx_r, yy_r, zz_r = generate_random_points_and_eval_z(sigma1, sigma2, sigma3)

# # 2 diag, 1 no diag
# xx_1, yy_1, zz_1 = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3)

# # 1 diag, 2 no diag
# xx_2, yy_2, zz_2 = generate_mesh_and_z(sigma1_d, sigma2, sigma3)
# # 3 no diag
# xx_3, yy_3, zz_3 = generate_mesh_and_z(sigma1, sigma2, sigma3)




X:  [[-2.         -1.91836735 -1.83673469 -1.75510204 -1.67346939 -1.59183673
  -1.51020408 -1.42857143 -1.34693878 -1.26530612 -1.18367347 -1.10204082
  -1.02040816 -0.93877551 -0.85714286 -0.7755102  -0.69387755 -0.6122449
  -0.53061224 -0.44897959 -0.36734694 -0.28571429 -0.20408163 -0.12244898
  -0.04081633  0.04081633  0.12244898  0.20408163  0.28571429  0.36734694
   0.44897959  0.53061224  0.6122449   0.69387755  0.7755102   0.85714286
   0.93877551  1.02040816  1.10204082  1.18367347  1.26530612  1.34693878
   1.42857143  1.51020408  1.59183673  1.67346939  1.75510204  1.83673469
   1.91836735  2.        ]
 [-2.         -1.91836735 -1.83673469 -1.75510204 -1.67346939 -1.59183673
  -1.51020408 -1.42857143 -1.34693878 -1.26530612 -1.18367347 -1.10204082
  -1.02040816 -0.93877551 -0.85714286 -0.7755102  -0.69387755 -0.6122449
  -0.53061224 -0.44897959 -0.36734694 -0.28571429 -0.20408163 -0.12244898
  -0.04081633  0.04081633  0.12244898  0.20408163  0.28571429  0.36734694
   0.4489

In [ ]:

fig = go.Figure(data=[go.Surface(z=zz.numpy(), x=xx, y=yy)])
fig.update_layout(scene=dict(aspectmode='data'))
fig.update_layout(scene=dict(camera=dict(eye=dict(x=2, y=2, z=1))))

#fig.show()

# Funcion para los sub-bloques

In [ ]:
def squeze_factor(Y):
  """
  Calculates a squeezing factor for a given set of values.

  Args:
  - Y (iterable): An iterable containing numeric values.

  Returns:
  - float: The squeezing factor. If the maximum value in Y is greater than 1, returns the reciprocal of the maximum value. Otherwise, returns 1.0.
  """
  e_f   = 0.0
  max_y = max(Y)
  if max_y > 1:
    e_f = 1/max_y
  else:
    e_f = 1.0
  return e_f

In [ ]:
class SubBlock:
    """
    Represents a sub-block in a 2D grid.

    Attributes:
    - vertices (np.ndarray): The vertices of the sub-block.
    - amplitude (int): The amplitude of the sub-block.
    - samples_inside (list): List of samples inside the sub-block.
    - output_values (list): List of output values.
    - index (list): List of index of the original point X

    Methods:
    - add_point(point): Add a point to the sub-block.
    """
    def __init__(self, amplitude=1, ista_layer=None):
        self.amplitude = amplitude
        self.ista_layer = ista_layer
        self._X  = []
        self._index  = []
        self.predicted_output = []
        self.output_values = []

    def add_point(self, point, y):
        self._X.append(point)
        self.output_values.append(y)


def get_sub_block_vertices(grid_size, row, col):
    """
    Get the vertices of a sub-block in a 2D grid.

    Args:
    - grid_size (int): The number of segments per dimension.
    - row (int): The row index of the sub-block.
    - col (int): The column index of the sub-block.

    Returns:
    np.ndarray: The vertices of the sub-block.
    """
    delta = 1.0 / grid_size
    x0 = col * delta
    x1 = (col + 1) * delta
    y0 = row * delta
    y1 = (row + 1) * delta
    return np.array([[x0, y0], [x1, y0], [x0, y1], [x1, y1]])


def locate_samples_in_sub_blocks(x_n, y, t, T):
    """
    Locate points in their respective sub-blocks in a 2D grid.

    Args:
    - x_n (np.ndarray): The normalized points between 0 and 1.
    - y (np.narray) : The output values associated with the samples
    - t (np.ndarray): The integer part of the normalized points.
    - T (int): The number of segments per dimension.

    Returns:
    np.ndarray: Array of SubBlock instances representing the sub-blocks.
    """

    sub_blocks = np.empty((T*T), dtype=object)

    for index in range(T*T):
          sub_blocks[index] = SubBlock()

    for i in range(x_n.shape[0]):
        point = x_n[i]
        location = t[i]
        sub_block = sub_blocks[location[0]*T + location[1]]
        sub_block.add_point(point, y[i])

    return sub_blocks

def generate_list_of_subblock(sub_blocks, l_functions):
  """
    Generate a list for all the sub-blocks with there expected squeze factor

    Arg:
      np.ndarray: Array of SubBlock instances representing the sub-blocks.

    Returns:
    list: List of all sub-block with there block.output_values modified.
  """
  list_sub_blocks = []
  for block in sub_blocks:
    print(f"OUTPUT VALUES: {len(block.output_values) }")
    if(len(block.output_values) != 0):
      block.amplitude = squeze_factor(block.output_values)
      block.ista_layer = ISTALayer(l_functions,SEED)
      block.output_values = [value * block.amplitude for value in block.output_values]

      list_sub_blocks.append(block)

  return list_sub_blocks


In [ ]:
def data_mapping(X, T):
    """
    Maps input data to a normalized range and separates it into integer and fractional parts.

    Args:
    - X (torch.Tensor): A tensor containing the input data.
    - T (int): The scaling factor to normalize the data.

    Returns:
    - t (numpy.ndarray): An array of integer parts of the normalized data.
    - x_n (torch.Tensor): A tensor of fractional parts of the normalized data.
    """
    delta = torch.max(X) - torch.min(X)
    eps   = delta*1e-6
    norm_x = ((X - torch.min(X)) / (delta + eps))*T
    t = norm_x.numpy().astype(int)
    x_n = norm_x - t
    return t, x_n


In [ ]:
def predict_on_test_set(X_test, model, T, train_sb):
    """
    Predicts on a test set using a given model and training sub-blocks.

    Args:
    - X_test (torch.Tensor): A tensor containing the test data.
    - model (Model): The model used for prediction.
    - T (int): The scaling factor for normalizing the data.
    - train_sb (list): A list of training sub-blocks used for prediction.

    Returns:
    - sorted_predictions (torch.Tensor): A tensor containing the sorted predictions for the test data.
    """
    t_test, x_n_test = data_mapping(X_test, T)

    sorted_predictions = torch.zeros(len(X_test), dtype=torch.float32)  # Tensor para almacenar las predicciones ordenadas

    for row in range(T):
        for col in range(T):
            sub_block_points = x_n_test[(t_test[:, 0] == row) & (t_test[:, 1] == col)]
            indices = np.where((t_test[:, 0] == row) & (t_test[:, 1] == col))[0]

            if len(sub_block_points) > 0:
                X_sub_block = torch.tensor(sub_block_points, dtype=torch.float32)

                try:
                    current_train_block = train_sb[row * T + col]
                    predictions_sub_block = model.predict(X_sub_block, current_train_block.ista_layer)
                    print("CURRENT ISTA on sub block ", current_train_block.ista_layer.h)
                    print("SUM VALUES H ", current_train_block.ista_layer.h.data.sum())
                    sorted_predictions[indices] = predictions_sub_block.float() / current_train_block.amplitude
                except IndexError:
                    # No se hace nada para los bloques dummy
                    pass

    return sorted_predictions


In [ ]:
def count_unique_combinations(T):
    """
    Count the unique combinations in the list generated by the data_mapping function.

    Args:
        T (int): Value T.

    Returns:
        dict: Dictionary with unique combinations as keys and the number of occurrences as values.
    """
    # Crear un diccionario para contar las combinaciones únicas
    conteo_combinaciones = defaultdict(int)

    # Contar las combinaciones únicas en T
    for combinacion in T:
      combinacion_tuple = tuple(combinacion)
      conteo_combinaciones[combinacion_tuple] += 1

    # Imprimir el conteo de combinaciones únicas
    for combinacion, cantidad in conteo_combinaciones.items():
      print(f"Combinación {combinacion}: {cantidad} veces")

# Configuracion y ejecución del modelo

In [86]:
def generate_uniform_sampling(total_points, n_samples=500, min_separation=1):
    """
    Generate uniform sampling indices with a minimum separation criterion.

    Args:
        total_points (int): Total number of data points.
        n_samples (int): Number of samples to generate (default is 500).
        min_separation (int): Minimum separation between selected indices (default is 1).

    Returns:
        list: List of selected indices.

    Example:
        sampled_indices = generate_uniform_sampling(1000, n_samples=200, min_separation=2)
        TODO> HACER ARRAY DE 1 A TOTAL POINTS, HACER RANDOM PERMUTE Y SACAR PRIMEROS 500
        random.permutation

    """
    np.random.seed(SEED)
    selected_indexes = np.random.permutation(total_points)[:n_samples]
    return selected_indexes

def sample_data(x_values, y_values, z_values, sampled_indices):
    """
    Sample data based on selected indices.

    Args:
        x_values (array-like): X-axis values.
        y_values (array-like): Y-axis values.
        z_values (array-like): Z-axis values.
        sampled_indices (list): List of indices to sample.

    Returns:
        tuple: Tuple containing sampled X (features) and y (labels).

    Example:
        X, y = sample_data(x_values, y_values, z_values, sampled_indices)
    """

    sampled_x = torch.tensor(x_values[sampled_indices], dtype=torch.float32)
    sampled_y = torch.tensor(y_values[sampled_indices], dtype=torch.float32)
    X = torch.stack((sampled_x, sampled_y), dim=1)
    y = torch.tensor(z_values[sampled_indices], dtype=torch.float32)
    return X, y

def build_model(n_samples, n_features, l_functions, eig_range, mu_range, vector_range, model_epochs, ista_epochs, ista_alpha, ista_lambd, mu_epochs, rho_epochs, dictionary_alpha, weight_decay):
    """
    Build and configure the SESM (Sparse Evolutionary Structural Modeling) model.

    Args:
    - n_samples (int): Number of training samples.
    - n_features (int): Number of features.
    - l_functions (int): Number of Gaussian functions.

    Returns:
    SESM_Model: An instance of the SESM model.
    """
    gaussian_function = GaussianFunctions(n_features=n_features, n_functions=l_functions,logger=logger, eig_range=eig_range, mu_range=mu_range, vector_range=vector_range,seed=SEED)
    model = SESM_Model(
        n_samples=n_samples,
        psi=gaussian_function,
        seed = SEED,
        model_epochs=model_epochs,
        ista_epochs=ista_epochs,
        ista_alpha=ista_alpha,
        ista_lambd=ista_lambd,
        mu_epochs=mu_epochs,
        rho_epochs=rho_epochs,
        dictionary_alpha=dictionary_alpha,
        weight_decay=weight_decay
    )
    return model

def train_model(model, X, y):
    """
    Train the SESM model using the specified training parameters.

    Args:
    - model (SESM_Model): The SESM model to be trained.
    - X (torch.Tensor): Input features.
    - y (torch.Tensor): Target values.

    Returns:
    None
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model.fit(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )

def plot_surface(train_dataset, test_dataset, X_train, y_train, Z, hypset, fngroup, iteration, losses_ISTA, losses_Dictionary):
    """
    Plots multiple subplots including loss curves, sampled data, original function, and surrogate model surface.

    Args:
    - train_dataset (dict): A dictionary containing the training dataset.
    - test_dataset (dict): A dictionary containing the test dataset.
    - X_train (torch.Tensor): The input data points for training.
    - y_train (torch.Tensor): The target values for training.
    - Z (torch.Tensor): The predicted values (surface) from the surrogate model.
    - hypset (str): The hyperparameter set identifier.
    - fngroup (str): The function group identifier.
    - iteration (int): The iteration number of the experiment.
    - losses_ISTA (list): List of ISTA losses over epochs.
    - losses_Dictionary (list): List of Dictionary losses over epochs.
    """
    fig = plt.figure(figsize=(15, 10))
    X = test_dataset["X"].reshape(50, 50)  # Remodelar X a una matriz 2D
    Y = test_dataset["Y"].reshape(50, 50)  # Remodelar Y a una matriz 2D
    Z = Z.clone().reshape(50, 50).detach().numpy()

    x_samples_train = X_train[:, 0]
    y_samples_train = X_train[:, 1]
    z_samples_train = y_train

    x_dataset = train_dataset["X"]
    y_dataset = train_dataset["Y"]
    z_dataset = train_dataset["Z"]

    #Total epochs = 6 [2 * ( 3 permutaciones )] * 16 bloques
    ax1 = fig.add_subplot(231)
    ax1.scatter(range(len(losses_ISTA)), losses_ISTA)
    ax1.set_xlabel('Total epochs')
    ax1.set_ylabel('losses_ISTA')
    ax1.set_title('losses_ISTA vs Total epochs')

    ax5 = fig.add_subplot(232)
    ax5.scatter(range(len(losses_Dictionary)), losses_Dictionary)
    ax5.set_xlabel('Total epochs')
    ax5.set_ylabel('losses_Dictionary')
    ax5.set_title('losses_Dictionary vs Total epochs')

    ax3 = fig.add_subplot(233)
    ax3.scatter(x_samples_train, y_samples_train)
    ax3.set_xlabel('X')
    ax3.set_ylabel('Y')
    ax3.set_title('Sampled Data')

    # Ajuste de los límites de los ejes
    ax3.set_xlim([min(x_samples_train), max(x_samples_train)])
    ax3.set_ylim([min(y_samples_train), max(y_samples_train)])

    ax4 = fig.add_subplot(234, projection='3d')
    ax4.plot_surface(X, Y, test_dataset["Z"].reshape(50, 50), cmap='viridis', alpha=0.9)
    ax4.scatter(x_samples_train, y_samples_train, z_samples_train, c='red')
    ax4.set_xlabel('X')
    ax4.set_ylabel('Y')
    ax4.set_zlabel('Z')
    ax4.set_title('Original Function')

    # Ajuste de los límites de los ejes
    ax4.set_xlim([min(x_samples_train), max(x_samples_train)])
    ax4.set_ylim([min(y_samples_train), max(y_samples_train)])
    ax4.set_zlim([min(z_samples_train), max(z_samples_train)])

    ax2 = fig.add_subplot(212, projection='3d')
    ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9)
    ax2.scatter(x_samples_train, y_samples_train, z_samples_train, c='red')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    ax2.set_title('Surrogate Model')

    # Ajuste de los límites de los ejes
    ax2.set_xlim([min(x_samples_train), max(x_samples_train)])
    ax2.set_ylim([min(y_samples_train), max(y_samples_train)])
    ax2.set_zlim([min(z_samples_train), max(z_samples_train)])

    filename = f"results_{hypset}/plots/{fngroup}.{iteration}.png"

    plt.tight_layout()
    plt.savefig(filename)
    plt.close(fig)



In [ ]:
def unstack_design_matrix(X_test):
    """
    Unstacks the design matrix into its individual components.

    Args:
    - X_test (torch.Tensor): A tensor containing the design matrix with multiple columns.

    Returns:
    - x_tensor (torch.Tensor): A tensor containing the first component (column) of the design matrix.
    - y_tensor (torch.Tensor): A tensor containing the second component (column) of the design matrix.
    """
    x_tensor, y_tensor = torch.unbind(X_test, dim=1)
    return x_tensor, y_tensor

In [ ]:
def create_design_matrix_train(xx, yy, zz, hyperparams):
    """
    Creates a design matrix for training by sampling data points.

    Args:
    - xx (numpy.ndarray): The x-coordinates of the grid.
    - yy (numpy.ndarray): The y-coordinates of the grid.
    - zz (numpy.ndarray): The z-values (target values) of the grid.
    - hyperparams (dict): A dictionary containing hyperparameters, including:
        - "n_samples" (int): The number of samples to generate.
        - "T" (int): The scaling factor for normalization.

    Returns:
    - X (numpy.ndarray): A 2D array containing the sampled x and y coordinates as the design matrix.
    - y (numpy.ndarray): A 1D array containing the sampled z-values as the target variable.
    """
    x_values = xx.ravel()
    y_values = yy.ravel()
    z_values = zz.ravel()

    n_samples = hyperparams["n_samples"]
    T = hyperparams["T"]

    total_points = len(x_values)

    sampled_indices = generate_uniform_sampling(total_points, n_samples=n_samples)
    X, y = sample_data(x_values, y_values, z_values, sampled_indices)

    return X, y

In [ ]:
def create_design_matrix_test(xx, yy, zz):
    """
    Creates a design matrix for testing from given grid coordinates and target values.

    Args:
    - xx (numpy.ndarray): The x-coordinates of the grid.
    - yy (numpy.ndarray): The y-coordinates of the grid.
    - zz (numpy.ndarray): The z-values (target values) of the grid.

    Returns:
    - X_test (torch.Tensor): A tensor containing the design matrix for testing, where each row represents a sample with x and y coordinates.
    - z_values (numpy.ndarray): A 1D array containing the target values (zz) flattened as a vector.
    """
    x_values = xx.ravel()
    y_values = yy.ravel()
    z_values = zz.ravel()

    x_tensor = torch.tensor(x_values)
    y_tensor = torch.tensor(y_values)
    X_test   =  torch.stack((x_tensor, y_tensor), dim=1)

    return X_test, z_values

In [ ]:
class ModeloSecuencial:
    def __init__(self, hyperparams, fngroup, iter, train_dataset, test_dataset, debug=True):
        self.hyperparams = hyperparams
        self.fngroup = fngroup
        self.iter = iter
        self.debug = debug
        self.model = self.build_model()
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset

    # Crear el modelo
    def build_model(self):
        n_samples = self.hyperparams["n_samples"]
        n_features = self.hyperparams["n_features"]
        l_functions = self.hyperparams["l_functions"]
        eig_range = self.hyperparams["eig_range"]
        mu_range = self.hyperparams["mu_range"]
        vector_range = self.hyperparams["vector_range"]
        model_epochs = self.hyperparams["m_epochs"]
        ista_epochs = self.hyperparams["h_epochs"]
        rho_epochs = self.hyperparams["rho_epochs"]
        mu_epochs = self.hyperparams["mu_epochs"]
        ista_alpha = self.hyperparams["ista_alpha"]
        ista_lambd = self.hyperparams["ista_lambd"]
        dictionary_alpha = self.hyperparams["dictionary_alpha"]
        weight_decay = self.hyperparams["weight_decay"]
        model = build_model(n_samples, n_features, l_functions, eig_range, mu_range, vector_range, model_epochs, ista_epochs, ista_alpha, ista_lambd, mu_epochs, rho_epochs, dictionary_alpha, weight_decay)
        return model

    # Entrenamiento estándar del modelo
    def train_regular(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        train_model(self.model, X, y)

    # Entrenamiento por sub-bloques del modelo
    def train_sequential(self, list_sub_blocks, T):
        permutation_times = self.hyperparams["permutation_times"]
        for _ in range(permutation_times):
            selected_indexes = np.random.permutation(T**2)
            permuted_list_sub_blocks = [list_sub_blocks[i] for i in selected_indexes]
            for block in permuted_list_sub_blocks:
                y = torch.tensor(block.output_values, dtype=torch.float32)
                X = torch.tensor(np.array(block._X), dtype=torch.float32)
                self.model.ista_layer = block.ista_layer
                train_model(self.model, X, y)

    # Predecir datos con el modelo estándar
    def predict_regular(self, X_test):
      print("type(X_test): ",type(X_test))
      y = self.model.predict(X_test, self.model.ista_layer)
      return y

    # Predecir datos con el modelo por sub-bloques
    def predict_sequential(self, X_test, list_sub_blocks):
      T = self.hyperparams["T"]
      t_test, x_n_test = data_mapping(X_test, T)
      y = predict_on_test_set(X_test, self.model, T, list_sub_blocks)
      return y

    # Evaluar el modelo de forma estándar
    def evaluate_regular(self, X_test, y_test):
        y = self.predict_regular(X_test)
        time = self.model.time / 60
        mse = mean_squared_error(y.clone().detach(), y_test)
        return y, time, mse

    # Evaluar el modelo por sub-bloques
    def evaluate_sequential(self, X_test, y_test, list_sub_blocks):
        y = self.predict_sequential(X_test, list_sub_blocks)
        time = self.model.time / 60
        mse = mean_squared_error(y.clone().detach(), y_test)
        return y, time, mse

    # Graficar el modelo
#    modelo_secuencial.plot_and_save_information(modelo_secuencial.train_dataset, modelo_secuencial.test_dataset, X_test, y_test, Z)
    def plot_and_save_information(self, train_dataset, test_dataset, X_train, y_train, Z):
      if self.hyperparams["n_features"] == 2:
        plot_surface(train_dataset, test_dataset, X_train, y_train, Z, self.hyperparams["hyp_set"], self.fngroup, self.iter, self.model.losses_ISTA, self.model.losses_Dictionary)

        df = pd.DataFrame({
            'Loss_ISTA': self.model.losses_ISTA,
            'Loss_Dictionary': self.model.losses_Dictionary
        })

        df.to_csv(f'results_{self.hyperparams["hyp_set"]}/stats/{self.iter}.csv', index=False)

        """for loss in model.losses:
          wandb.log({"loss": loss})"""
      else:
        print("n_features > 3")

In [85]:
# Ejecutar el experimento
# Cambiar los datos por una matriz de matriz de diseño X
# Un condicional para una grafica de > dimensiones
# run_experiment deberia de estar fuera de esta clase

def run_experiment(X_train, y_train, X_test, y_test, modelo_secuencial):
  """
  Runs an experiment using a sequential or regular training mode with a sequential model.

  Args:
  - X_train (torch.Tensor): The training input data.
  - y_train (torch.Tensor): The training target values.
  - X_test (torch.Tensor): The test input data.
  - y_test (torch.Tensor): The test target values.
  - modelo_secuencial (SequentialModel): The sequential model used for training and evaluation.

  Returns:
  - time (float): The time taken for the experiment.
  - mse_value (float): The mean squared error value obtained from the experiment.
  """
  T = modelo_secuencial.hyperparams["T"]
  l_functions = modelo_secuencial.hyperparams["l_functions"]

  t, x_n = data_mapping(X_train, T)
  count_unique_combinations(t)

  sub_blocks = locate_samples_in_sub_blocks(x_n, y_train, t, T)

  list_sub_blocks = generate_list_of_subblock(sub_blocks, l_functions)

  time, mse_value = 0, 0

  # Entrenamiento secuencial o por sub-bloques
  if modelo_secuencial.hyperparams["mode"] == "regular":
      modelo_secuencial.train_regular(X_train, y_train)
      Z, time, mse_value = modelo_secuencial.evaluate_regular(X_test, y_test)
      # Graficar
      modelo_secuencial.plot_and_save_information(modelo_secuencial.train_dataset, modelo_secuencial.test_dataset, X_train, y_train, Z)
      #modelo_secuencial.plot_and_save_information( X_train, y_train, X_test, y_test, Z, modelo_secuencial.train_dataset)

  elif modelo_secuencial.hyperparams["mode"] == "secuencial":
      modelo_secuencial.train_sequential(list_sub_blocks, T)
      Z, time, mse_value = modelo_secuencial.evaluate_sequential(X_test, y_test, list_sub_blocks)
      # Graficar
      modelo_secuencial.plot_and_save_information(modelo_secuencial.train_dataset, modelo_secuencial.test_dataset, X_train, y_train, Z)

  return time, mse_value

In [ ]:

def plot_stats(directory, num_files):
    """
    Plot statistics for loss values from multiple CSV files.

    Args:
        directory (str): The directory containing CSV files.
        num_files (int): The number of CSV files to process.

    Returns:
        None: Displays an interactive plot and saves an HTML file.

    Note:
        Assumes that each CSV file contains loss values for the same number of epochs.

    """
    fig = px.scatter(title='Loss analysis')
    m_epochs_losses = []

    for i in range(num_files):
        file_path = f"{directory}/stats/{i}.csv"

        df = pd.read_csv(file_path)
        m_epochs_losses.append(df)

    merged_losses = pd.concat(m_epochs_losses, axis=1)

    # Compute mean, std, min, and max for each row
    summary_df = pd.DataFrame({
        'Mean': merged_losses.mean(axis=1),
        'Std': merged_losses.std(axis=1),
        'Min': merged_losses.min(axis=1),
        'Max': merged_losses.max(axis=1)
    })

    summary_df.to_csv(f'{directory}/stats/processed.csv', index=False)

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Mean'],
        mode='lines+markers',
        error_y=dict(type='data', array=summary_df['Std']),
        name='Mean'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Max'],
        mode='markers',
        name='Max'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Min'],
        mode='markers',
        name='Min'
    )

    fig.update_layout(
        xaxis_title='m_epochs',
        yaxis_title='Loss',
        legend_title='Legend',
        showlegend=True
    )
    fig.show()
    fig.write_html(f"interactive_plot-{directory}.html")

In [ ]:
import csv

def save_results(data, fngroup):
  # Compute Mean and Std for executio
  times = [item[1] for item in data]
  mse_values = [item[2] for item in data]

  average_time = np.mean(times)
  std_time = np.std(times)
  average_mse = np.mean(mse_values)
  std_mse = np.std(mse_values)

  with open(f"results_{fngroup}.csv", mode="w", newline="") as file:
      writer = csv.writer(file)
      writer.writerow(["Repetion", "Time (min)", "MSE"])
      writer.writerows(data)
      writer.writerow(["Mean", average_time, average_mse])
      writer.writerow(["Std", std_time, std_mse])


# Nomenclatura de experimentos

<Set de hiperparámetros>. <Set de datos (conjunto de gaussianas)>.<Número de repetición del experimento>


## Set de Hiperparámetros
|  Hiperparámetro | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| n_samples       | 50            | 100           | 500           |
| n_features      | 2             | 2             | 2             |
| l_functions     | 20            | 6             | 10            |
| ista_alpha      | 0.06          | 0.0125        | 0.0125        |
| ista_lambd      | 0.005         | 0.001         | 0.001         |
| dictionary_alpha| 0.06          | 0.0125        | 0.0125        |
| m_epochs        | 25            | 500           | 300           |
| dict_epochs     | 800           | 20            | 60            |
| h_epochs        | 1000          | 50            | 100           |


### Set de datos

|     Set      | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| Gaussianas 1    | Exp 1.1.x     | Exp 2.1.x     | Exp 3.1.x     |
| Gaussianas 2    | Exp 1.2.x     | Exp 2.2.x     | Exp 3.2.x     |
| Gaussianas 3    | Exp 1.3.x     | Exp 2.3.x     | Exp 3.3.x     |
| Gaussianas 4    | Exp 1.4.x     | Exp 2.4.x     | Exp 3.4.x     |


In [ ]:
N_iter = 10 #
experiment_3 = {
      "hyp_set": 1,
      "n_samples"	: 500,
      "n_features" : 2,
      "l_functions":  100,
      "eig_range": [1e0, 1e1],
      "mu_range": [-2, 2],
      "vector_range": [1e0, 1e1],
      "ista_alpha"	: 0.05502,
      "ista_lambd"	 : 0.01007,
      "dictionary_alpha": 0.08928,
      "rho_epochs": 10,
      "mu_epochs": 10,
      "m_epochs" : 10,
      "dict_epochs": 10,
      "h_epochs": 10,
      "T": 4,
      "weight_decay": 0.004875,
      "permutation_times": 3,
      "mode": "secuencial"
}
#1, 2, 3 --> m_epochs


# Correr experimentos

In [ ]:
# 1- Meter integrar trainDataset con el codigo de run experiment para graficar.
# 2- Evaluar en un modelo secuencial y regular
# 3- Comentar codigo

data = []

trainDataset = {"X": xx_r.ravel(), "Y": yy_r.ravel(), "Z": zz_r.ravel() }
testDataset = {"X": xx.ravel(), "Y": yy.ravel(), "Z": zz.ravel() }

# Crea una instancia de la clase ModeloSecuencial
modelo = ModeloSecuencial(experiment_3, fngroup=1, iter=0, train_dataset=trainDataset, test_dataset=testDataset, debug=True)


# Crear la matriz de diseño
X_train, y_train = create_design_matrix_train(xx_r, yy_r, zz_r, experiment_3)


# Crear la matriz de diseño
X_test, y_test = create_design_matrix_test(xx, yy, zz)


for i in range(N_iter):
    """
    wandb.init(
        # Set the project where this run will be logged
        project="SESM-2D-GaussianFunction",
        # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
        name=f"12-11-2023-experiments-{i}",
        # Track hyperparameters and run metadata
        config = experiment_3
    )
    """
    modelo.iter = i
    # Ejecuta el experimento
    time, mse = run_experiment(X_train, y_train, X_test, y_test, modelo)

    # Almacena los resultados
    data.append((i, time, mse))

    # wandb.finish()

# Guarda los resultados
save_results(data=data, fngroup=1)

INFO:logger:Shape of Mu: torch.Size([2, 100])
INFO:logger:Shape of Rho: torch.Size([3, 100])
INFO:logger:Shape of Theta: torch.Size([5, 100])
<ipython-input-58-b2d163f7e0f3>:43: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



Combinación (0, 2): 33 veces
Combinación (1, 1): 38 veces
Combinación (0, 1): 44 veces
Combinación (1, 0): 24 veces
Combinación (2, 3): 33 veces
Combinación (3, 3): 38 veces
Combinación (1, 2): 30 veces
Combinación (3, 2): 32 veces
Combinación (0, 3): 30 veces
Combinación (3, 0): 25 veces
Combinación (3, 1): 28 veces
Combinación (1, 3): 35 veces
Combinación (0, 0): 23 veces
Combinación (2, 0): 30 veces
Combinación (2, 1): 26 veces
Combinación (2, 2): 31 veces
OUTPUT VALUES: 23
OUTPUT VALUES: 44
OUTPUT VALUES: 33
OUTPUT VALUES: 30
OUTPUT VALUES: 24
OUTPUT VALUES: 38
OUTPUT VALUES: 30
OUTPUT VALUES: 35
OUTPUT VALUES: 30
OUTPUT VALUES: 26
OUTPUT VALUES: 31
OUTPUT VALUES: 33
OUTPUT VALUES: 25
OUTPUT VALUES: 28
OUTPUT VALUES: 32
OUTPUT VALUES: 38
Epoch 1 Loss_ISTA: 0.00026522771804593503 Loss_Dictionary: 0.008814078755676746 

Epoch 2 Loss_ISTA: 0.00025485572405159473 Loss_Dictionary: 0.0002641544851940125 

Epoch 3 Loss_ISTA: 0.000277906161500141 Loss_Dictionary: 0.000259718595771119 

Epo

<ipython-input-56-9a3018bb36df>:24: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



Combinación (0, 2): 33 veces
Combinación (1, 1): 38 veces
Combinación (0, 1): 44 veces
Combinación (1, 0): 24 veces
Combinación (2, 3): 33 veces
Combinación (3, 3): 38 veces
Combinación (1, 2): 30 veces
Combinación (3, 2): 32 veces
Combinación (0, 3): 30 veces
Combinación (3, 0): 25 veces
Combinación (3, 1): 28 veces
Combinación (1, 3): 35 veces
Combinación (0, 0): 23 veces
Combinación (2, 0): 30 veces
Combinación (2, 1): 26 veces
Combinación (2, 2): 31 veces
OUTPUT VALUES: 23
OUTPUT VALUES: 44
OUTPUT VALUES: 33
OUTPUT VALUES: 30
OUTPUT VALUES: 24
OUTPUT VALUES: 38
OUTPUT VALUES: 30
OUTPUT VALUES: 35
OUTPUT VALUES: 30
OUTPUT VALUES: 26
OUTPUT VALUES: 31
OUTPUT VALUES: 33
OUTPUT VALUES: 25
OUTPUT VALUES: 28
OUTPUT VALUES: 32
OUTPUT VALUES: 38
Epoch 1 Loss_ISTA: 0.014366482384502888 Loss_Dictionary: 0.04209241271018982 

Epoch 2 Loss_ISTA: 0.00842172559350729 Loss_Dictionary: 0.01334668230265379 

Epoch 3 Loss_ISTA: 0.005931235384196043 Loss_Dictionary: 0.008018694818019867 

Epoch 4 Los

KeyboardInterrupt: 

In [ ]:
data_1 = []

for i in range(N_iter):
  """wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
    # Crea una instancia de la clase ModeloSecuencial para cada iteración
    modelo = ModeloSecuencial(experiment_3, fngroup=2, iter=i, debug=True)

    # Ejecuta el experimento xx_r,yy_r,zz_r,xx,yy,zz
    time, mse = modelo.run_experiment(xx_1, yy_1, zz_1, xx, yy, zz)

    # Almacena los resultados
    data_1.append((i, time, mse))

    # wandb.finish()

save_results(data=data_1, fngroup=2)

In [ ]:
#Unit test de la inicialización únicamente
stats_dir = f'results_{experiment_3["hyp_set"]}'
plot_stats(stats_dir, N_iter)

In [ ]:
data_2 = []
for i in range(N_iter):
  """  wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  |# Crea una instancia de la clase ModeloSecuencial para cada iteración
    modelo = ModeloSecuencial(experiment_3, fngroup=3, iter=i, debug=True)

    # Ejecuta el experimento xx_r,yy_r,zz_r,xx,yy,zz
    time, mse = modelo.run_experiment(xx_2, yy_2, zz_2, xx, yy, zz)

    # Almacena los resultados
    data_2.append((i, time, mse))

    # wandb.finish()

save_results(data=data_2, fngroup=3)

In [ ]:
data_3 = []
for i in range(N_iter):
  """  wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
    # Crea una instancia de la clase ModeloSecuencial para cada iteración
    modelo = ModeloSecuencial(experiment_3, fngroup=4, iter=i, debug=True)

    # Ejecuta el experimento xx_r,yy_r,zz_r,xx,yy,zz
    time, mse = modelo.run_experiment(xx_3, yy_3, zz_3, xx, yy, zz)

    # Almacena los resultados
    data_3.append((i, time, mse))

    # wandb.finish()

save_results(data=data_3, fngroup=4)

In [ ]:
files.download('results_1.csv')
!zip -r results_1.zip results_1/
files.download('results_1.zip')

In [ ]:
#files.download('results_2.csv')
!zip -r results_4.zip results_4/
files.download('results_4.zip')

In [ ]:
# files.download('results_3.csv')
!zip -r results_3.zip results_3/
files.download('results_3.zip')